# MITRE ATT&CK Knowledge Graph — v2.0
**Full pipeline:** Tactics → Techniques → Sub-techniques → Groups → Software → CAPEC → CWE → CVE

## Graph schema
```
(CVE)-[:CAUSE_WEAKNESS]->(CWE)
(CWE)<-[:TARGETS_WEAKNESS]-(CAPEC)
(CAPEC)-[:MAPS_TO_TECHNIQUE]->(Technique)
(Technique)-[:BELONGS_TO]->(Tactic)
(Technique)-[:HAS_SUBTECHNIQUE]->(Technique)   # sub-techniques
(Group)-[:USES]->(Technique)
(Group)-[:USES]->(Software)
(Software)-[:USES]->(Technique)
(CAPEC)-[:CHILD_OF]->(CAPEC)                   # CAPEC hierarchy
```

## Run order
1. Cell 1 – Constraints & indexes (run once)
2. Cell 2 – Ingest ATT&CK (Tactics + Techniques + Sub-techniques + Groups + Software)
3. Cell 3 – Ingest CAPEC nodes + hierarchy + CWE links + ATT&CK links
4. Cell 4 – Ingest CVEs (NVD) with CWE bridge
5. Cell 5 – Verify graph health


---
## Cell 1 — Config, driver, constraints

In [30]:
from neo4j import GraphDatabase

URI  = "bolt://localhost:7687"
AUTH = ("neo4j", "kg_mitre_v1.1")   # ← change if needed

driver = GraphDatabase.driver(URI, auth=AUTH)

CONSTRAINTS = [
    # Node uniqueness — prevents duplicates on re-runs
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Technique)  REQUIRE t.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (ta:Tactic)    REQUIRE ta.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (g:Group)      REQUIRE g.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Software)   REQUIRE s.stix_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:CAPEC)      REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (w:CWE)        REQUIRE w.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (v:CVE)        REQUIRE v.id IS UNIQUE",
    # Indexes for fast lookups
    "CREATE INDEX IF NOT EXISTS FOR (t:Technique)  ON (t.mitre_id)",
    "CREATE INDEX IF NOT EXISTS FOR (ta:Tactic)    ON (ta.mitre_id)",
]

with driver.session() as s:
    for stmt in CONSTRAINTS:
        s.run(stmt)

print("✅ Constraints and indexes ready.")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
✅ Constraints and indexes ready.


---
## Cell 2 — Ingest ATT&CK STIX bundle
Ingests: **Tactics**, **Techniques**, **Sub-techniques**, **Groups**, **Software (malware + tools)**  
Relationships: `BELONGS_TO`, `HAS_SUBTECHNIQUE`, `USES`

In [32]:
import json
from stix2 import MemoryStore, Filter

ATTACK_FILE = 'enterprise-attack.json' 

def _get(obj, key, default=None):
    """Unified getter: works for both stix2 objects and plain dicts."""
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _id(obj):
    """Get the STIX id field from either a stix2 object or a dict."""
    return obj['id'] if isinstance(obj, dict) else obj.id

def _name(obj):
    return obj['name'] if isinstance(obj, dict) else obj.name

def mitre_id(obj):
    """Extract the MITRE ATT&CK external ID (e.g. T1059, TA0112, G0016)."""
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('external_id') if isinstance(ref, dict) else getattr(ref, 'external_id', None)
    return _id(obj)  # fallback to STIX id

def url(obj):
    refs = _get(obj, 'external_references', [])
    for ref in refs:
        src = ref.get('source_name') if isinstance(ref, dict) else getattr(ref, 'source_name', '')
        if src == 'mitre-attack':
            return ref.get('url', '') if isinstance(ref, dict) else getattr(ref, 'url', '')
    return ''

def run_full_attack_ingestion(file_path):
    print("Loading STIX bundle...")
    with open(file_path) as f:
        bundle = json.load(f)
    store = MemoryStore(stix_data=bundle)

    with driver.session() as s:
        # ── 1. TACTICS (x-mitre-tactic) ──────────────────────────────────────
        print("Ingesting Tactics...")
        tactics = store.query([Filter('type', '=', 'x-mitre-tactic')])
        for t in tactics:
            mid = mitre_id(t)
            s.run("""
                MERGE (ta:Tactic {stix_id: $stix_id})
                SET ta.mitre_id   = $mid,
                    ta.name       = $name,
                    ta.short_name = $short,
                    ta.description= $desc,
                    ta.url        = $url
            """, stix_id=_id(t), mid=mid, name=_name(t),
                 short=_get(t, 'x_mitre_shortname', ''),
                 desc=_get(t, 'description', ''), url=url(t))
        print(f"  → {len(tactics)} tactics ingested (including TA0112)")

        # ── 2. TECHNIQUES + SUB-TECHNIQUES (attack-pattern) ──────────────────
        print("Ingesting Techniques & Sub-techniques...")
        techniques = store.query([Filter('type', '=', 'attack-pattern')])
        # Filter out revoked/deprecated
        techniques = [t for t in techniques
                      if not t.get('revoked') and not t.get('x_mitre_deprecated')]
        for t in techniques:
            mid = mitre_id(t)
            is_sub = '.' in mid
            platforms = t.get('x_mitre_platforms', [])
            s.run("""
                MERGE (tech:Technique {stix_id: $stix_id})
                SET tech.mitre_id    = $mid,
                    tech.name        = $name,
                    tech.description = $desc,
                    tech.is_subtechnique = $is_sub,
                    tech.platforms   = $platforms,
                    tech.url         = $url
            """, stix_id=t.id, mid=mid, name=t.name,
                 desc=t.get('description', ''), is_sub=is_sub,
                 platforms=platforms, url=url(t))
        print(f"  → {len(techniques)} techniques/sub-techniques")

        # ── 3. GROUPS (intrusion-set) ─────────────────────────────────────────
        print("Ingesting Groups (Threat Actors)...")
        groups = store.query([Filter('type', '=', 'intrusion-set')])
        groups = [g for g in groups
                  if not g.get('revoked') and not g.get('x_mitre_deprecated')]
        for g in groups:
            mid = mitre_id(g)
            aliases = g.get('aliases', [])
            s.run("""
                MERGE (gr:Group {stix_id: $stix_id})
                SET gr.mitre_id   = $mid,
                    gr.name       = $name,
                    gr.aliases    = $aliases,
                    gr.description= $desc,
                    gr.url        = $url
            """, stix_id=g.id, mid=mid, name=g.name,
                 aliases=aliases, desc=g.get('description', ''), url=url(g))
        print(f"  → {len(groups)} groups")

        # ── 4. SOFTWARE (malware + tool) ──────────────────────────────────────
        print("Ingesting Software (Malware + Tools)...")
        software = (
            store.query([Filter('type', '=', 'malware')]) +
            store.query([Filter('type', '=', 'tool')])
        )
        software = [sw for sw in software
                    if not sw.get('revoked') and not sw.get('x_mitre_deprecated')]
        for sw in software:
            mid = mitre_id(sw)
            s.run("""
                MERGE (so:Software {stix_id: $stix_id})
                SET so.mitre_id   = $mid,
                    so.name       = $name,
                    so.type       = $sw_type,
                    so.description= $desc,
                    so.url        = $url
            """, stix_id=sw.id, mid=mid, name=sw.name,
                 sw_type=sw.type, desc=sw.get('description', ''), url=url(sw))
        print(f"  → {len(software)} software items")

        # ── 5. RELATIONSHIPS & PROCEDURES ────────────────────────────────────
        print("Building relationships & isolating structural procedures...")
        rels = store.query([Filter('type', '=', 'relationship')])

        tactic_phase_map = {_get(t, 'x_mitre_shortname'): _id(t) for t in tactics}

        tactic_count = 0
        subtechnique_count = 0
        procedure_count = 0

        # 5a. Technique → Tactic (via kill_chain_phases on each technique)
        for t in techniques:
            for phase in t.get('kill_chain_phases', []):
                short = phase.get('phase_name')
                ta_stix = tactic_phase_map.get(short)
                if ta_stix:
                    s.run("""
                        MATCH (tech:Technique {stix_id: $tech_id})
                        MATCH (ta:Tactic {stix_id: $ta_id})
                        MERGE (tech)-[:BELONGS_TO]->(ta)
                    """, tech_id=t.id, ta_id=ta_stix)
                    tactic_count += 1

        # 5b. Sub-technique → Parent technique
        for rel in rels:
            if rel.relationship_type == 'subtechnique-of':
                s.run("""
                    MATCH (parent:Technique {stix_id: $parent_id})
                    MATCH (sub:Technique   {stix_id: $sub_id})
                    MERGE (parent)-[:HAS_SUBTECHNIQUE]->(sub)
                """, parent_id=rel.target_ref, sub_id=rel.source_ref)
                subtechnique_count += 1

        # 5c. Group/Software USES Technique/Software -> Elevated to Procedure Nodes
        for rel in rels:
            if rel.relationship_type == 'uses':
                proc_desc = _get(rel, 'description', '')
                
                # Derive a clean, distinct identifier string for the intermediate node
                proc_stix_id = f"procedure--{_id(rel).split('--')[-1]}"
                
                s.run("""
                    MATCH (src {stix_id: $src})
                    MATCH (tgt {stix_id: $tgt})
                    
                    // Instantiate the intermediate Procedure context node
                    MERGE (p:Procedure {stix_id: $proc_stix_id})
                    SET p.description = $desc,
                        p.url         = $url
                        
                    // Build structural tracking paths
                    MERGE (src)-[:EXECUTED]->(p)
                    MERGE (p)-[:TARGETS]->(tgt)

                    // NEW: Connect Group directly to Software if src is a Group and tgt is Software
                    WITH src, tgt
                    WHERE src:Group AND tgt:Software
                    MERGE (src)-[:USES]->(tgt)
                """, src=rel.source_ref, tgt=rel.target_ref, 
                     proc_stix_id=proc_stix_id, desc=proc_desc, url=url(rel))
                procedure_count += 1

        print(f"  → {tactic_count} BELONGS_TO (Technique→Tactic)")
        print(f"  → {subtechnique_count} HAS_SUBTECHNIQUE relationships created")
        print(f"  → {procedure_count} Procedure nodes mapped via (:EXECUTED) and (:TARGETS)")

    print("\n✅ ATT&CK Knowledge Graph Pipeline Execution Complete!")

run_full_attack_ingestion(ATTACK_FILE)

Loading STIX bundle...
Ingesting Tactics...
  → 14 tactics ingested (including TA0112)
Ingesting Techniques & Sub-techniques...
  → 691 techniques/sub-techniques
Ingesting Groups (Threat Actors)...
  → 172 groups
Ingesting Software (Malware + Tools)...
  → 784 software items
Building relationships & isolating structural procedures...
  → 887 BELONGS_TO (Technique→Tactic)
  → 477 HAS_SUBTECHNIQUE relationships created
  → 17270 Procedure nodes mapped via (:EXECUTED) and (:TARGETS)

✅ ATT&CK Knowledge Graph Pipeline Execution Complete!


In [37]:
import requests
import json
import time
from datetime import datetime, timedelta

# ───────────────────────────────────────────────────────────────────────
# 1. HARVEST CWEs FROM NVD (Bulk Year-Filtered Loop)
# ───────────────────────────────────────────────────────────────────────
def stream_nvd_with_cwes(driver, start_year=2015):
    base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"User-Agent": "CyberKnowledgeGraphEngine/2.0"}
    
    current_start = datetime(start_year, 1, 1)
    end_date_final = datetime.now()
    window_days = 120

    while current_start < end_date_final:
        current_end = min(current_start + timedelta(days=window_days), end_date_final)
        start_str = current_start.strftime('%Y-%m-%dT00:00:00.000Z')
        end_str = current_end.strftime('%Y-%m-%dT23:59:59.999Z')
        
        print(f"📅 NVD Window: {current_start.strftime('%Y-%m-%d')} to {current_end.strftime('%Y-%m-%d')}")
        start_index = 0
        results_per_page = 2000
        
        while True:
            params = {"pubStartDate": start_str, "pubEndDate": end_str, "resultsPerPage": results_per_page, "startIndex": start_index}
            try:
                time.sleep(1.0) # Prevent 403 blocks
                response = requests.get(base_url, params=params, headers=headers, timeout=30)
                if response.status_code != 200: break
                
                data = response.json()
                vulnerabilities = data.get('vulnerabilities', [])
                if not vulnerabilities: break
                
                batch_payload = []
                for v in vulnerabilities:
                    cve_obj = v.get('cve', {})
                    cve_id = cve_obj.get('id')
                    desc = next((d.get('value') for d in cve_obj.get('descriptions', []) if d.get('lang') == 'en'), '')
                    
                    # EXTRACT CWEs
                    cwes = []
                    for weakness in cve_obj.get('weaknesses', []):
                        for desc_obj in weakness.get('description', []):
                            val = desc_obj.get('value')
                            if val and val.startswith('CWE-') and val != 'CWE-noinfo':
                                cwes.append(val)
                    
                    batch_payload.append({"cve_id": cve_id, "description": desc, "cwes": cwes})
                
                if batch_payload:
                    with driver.session() as session:
                        session.run("""
                            UNWIND $batch AS record
                            MERGE (c:CVE {id: record.cve_id})
                            SET c.description = record.description, c.source = 'NVD'
                            WITH c, record
                            UNWIND record.cwes AS cwe_id
                            MERGE (w:CWE {id: cwe_id})
                            MERGE (c)-[:IDENTIFIES_WEAKNESS]->(w)
                        """, batch=batch_payload)
                
                if len(vulnerabilities) < results_per_page: break
                start_index += results_per_page
            except Exception:
                break
        current_start = current_end + timedelta(days=1)

# ───────────────────────────────────────────────────────────────────────
# 2. HARVEST CWEs FROM CISA KEV (String Splitting Parser)
# ───────────────────────────────────────────────────────────────────────
def ingest_cisa_with_cwes(driver):
    print("📥 Syncing CISA KEV + Split CWE Pipeline...")
    url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"
    vulnerabilities = requests.get(url).json().get('vulnerabilities', [])
    
    payload = []
    for v in vulnerabilities:
        cve_id = v.get('cveID')
        desc = v.get('shortDescription')
        
        # EXTRACT & CLEAN CWEs (Handles pipe separation like 'CWE-22|CWE-434')
        raw_cwes = v.get('relatedCWEs', '')
        cwes = [c.strip() for c in raw_cwes.split('|') if c.strip().startswith('CWE-') and c.strip() != 'CWE-noinfo']
        
        payload.append({"cve_id": cve_id, "desc": desc, "cwes": cwes})
        
    with driver.session() as session:
        session.run("""
            UNWIND $batch AS item
            MERGE (c:CVE {id: item.cve_id})
            SET c.is_actively_exploited = true,
                c.description = CASE WHEN c.description IS NULL OR c.description = '' THEN item.desc ELSE c.description END
            
            WITH c, item
            UNWIND item.cwes AS cwe_id
            MERGE (w:CWE {id: cwe_id})
            MERGE (c)-[:IDENTIFIES_WEAKNESS]->(w)
        """, batch=payload)
    print("✅ CISA CWE link synchronization complete.")

# ───────────────────────────────────────────────────────────────────────
# 3. HARVEST CWEs FROM OSV (Ecosystem Advisory Inspector)
# ───────────────────────────────────────────────────────────────────────
def enrich_osv_with_cwes(driver, ecosystem, package_names):
    print(f"📦 Syncing OSV Ecosystem CWE paths for {ecosystem}...")
    url = "https://api.osv.dev/v1/query"
    payload_batch = []
    
    for pkg in package_names:
        try:
            res = requests.post(url, json={"package": {"name": pkg, "ecosystem": ecosystem}}, timeout=10)
            if res.status_code != 200: continue
            
            for v in res.json().get('vulns', []):
                summary = v.get('summary', '')
                
                # EXTRACT CWEs from OSV/GHSA inner metadata block structures
                cwes = []
                # Check for standard database_specific mapping blocks
                db_specific = v.get('database_specific', {})
                for cwe_elem in db_specific.get('cwe_ids', []):
                    if cwe_elem.startswith('CWE-'):
                        cwes.append(cwe_elem)
                
                for alias in v.get('aliases', []):
                    if alias.startswith("CVE-"):
                        payload_batch.append({"cve_id": alias, "summary": summary, "cwes": cwes})
        except Exception:
            continue

    if payload_batch:
        with driver.session() as session:
            session.run("""
                UNWIND $batch AS item
                MERGE (c:CVE {id: item.cve_id})
                SET c.description = CASE WHEN c.description IS NULL OR c.description = '' THEN item.summary ELSE c.description END
                
                WITH c, item
                UNWIND item.cwes AS cwe_id
                MERGE (w:CWE {id: cwe_id})
                MERGE (c)-[:IDENTIFIES_WEAKNESS]->(w)
            """, batch=payload_batch)
    print(f"✅ OSV Ecosystem CWE linkage complete.")

if __name__ == "__main__":
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    # Run the consolidated ingestion passes sequentially
    stream_nvd_with_cwes(driver, start_year=2015)
    ingest_cisa_with_cwes(driver)
    enrich_osv_with_cwes(driver, "Maven", ["org.apache.logging.log4j:log4j-core"])
    
    driver.close()

📅 NVD Window: 2015-01-01 to 2015-05-01
📅 NVD Window: 2015-05-02 to 2015-08-30
📅 NVD Window: 2015-08-31 to 2015-12-29
📅 NVD Window: 2015-12-30 to 2016-04-28
📅 NVD Window: 2016-04-29 to 2016-08-27
📅 NVD Window: 2016-08-28 to 2016-12-26
📅 NVD Window: 2016-12-27 to 2017-04-26
📅 NVD Window: 2017-04-27 to 2017-08-25
📅 NVD Window: 2017-08-26 to 2017-12-24
📅 NVD Window: 2017-12-25 to 2018-04-24
📅 NVD Window: 2018-04-25 to 2018-08-23
📅 NVD Window: 2018-08-24 to 2018-12-22
📅 NVD Window: 2018-12-23 to 2019-04-22
📅 NVD Window: 2019-04-23 to 2019-08-21
📅 NVD Window: 2019-08-22 to 2019-12-20
📅 NVD Window: 2019-12-21 to 2020-04-19
📅 NVD Window: 2020-04-20 to 2020-08-18
📅 NVD Window: 2020-08-19 to 2020-12-17
📅 NVD Window: 2020-12-18 to 2021-04-17
📅 NVD Window: 2021-04-18 to 2021-08-16
📅 NVD Window: 2021-08-17 to 2021-12-15
📅 NVD Window: 2021-12-16 to 2022-04-15
📅 NVD Window: 2022-04-16 to 2022-08-14
📅 NVD Window: 2022-08-15 to 2022-12-13
📅 NVD Window: 2022-12-14 to 2023-04-13
📅 NVD Window: 2023-04-14 

total CISA CVE - 1599. out of 1599, 1583 has no relation with CVE.


to fill this gap, one way can be establish a connection between CISA and NVD

In [ ]:
def fast_track_cisa_nvd_sync():
    """Finds all CISA CVEs in the graph missing NVD data and updates them directly."""
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    # Step 1: Pull target CVE IDs from your current graph state
    with driver.session() as session:
        result = session.run("""
            MATCH (c:CVE)
            WHERE c.source_cisa = 'CISA_KEV' AND (c.source IS NULL OR c.source <> 'NVD_API_Bulk')
            RETURN c.id AS cve_id
        """)
        cisa_cve_ids = [row["cve_id"] for row in result]
        
    print(f"🎯 Found {len(cisa_cve_ids)} unmapped CISA CVEs. Running fast-track sync...")
    
    base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0?cveId="
    headers = {"User-Agent": "CyberKnowledgeGraphEngine/2.0"}
    
    # Step 2: Loop directly over just these specific IDs
    for cve_id in cisa_cve_ids:
        try:
            time.sleep(0.6)  # Safe delay to prevent public API throttling
            response = requests.get(f"{base_url}{cve_id}", headers=headers, timeout=15)
            
            if response.status_code != 200:
                continue
                
            data = response.json()
            vulnerabilities = data.get('vulnerabilities', [])
            if not vulnerabilities:
                continue
                
            cve_obj = vulnerabilities[0].get('cve', {})
            desc = next((d.get('value') for d in cve_obj.get('descriptions', []) if d.get('lang') == 'en'), '')
            
            cwes = []
            for weakness in cve_obj.get('weaknesses', []):
                for desc_obj in weakness.get('description', []):
                    val = desc_obj.get('value')
                    if val and val.startswith('CWE-') and val != 'CWE-noinfo':
                        cwes.append(val)
                        
            # Step 3: Update the node inside your graph instantly
            with driver.session() as session:
                session.run("""
                    MATCH (c:CVE {id: $cve_id})
                    SET c.description = $desc,
                        c.source = 'NVD_API_Bulk'
                    WITH c
                    UNWIND $cwes AS cwe_id
                    MERGE (w:CWE {id: cwe_id})
                    MERGE (c)-[:IDENTIFIES_WEAKNESS]->(w)
                """, cve_id=cve_id, desc=desc, cwes=cwes)
            print(f"  ✅ Synchronized {cve_id} -> Added {len(cwes)} CWE paths.")
            
        except Exception as e:
            print(f"  ❌ Failed sync for {cve_id}: {e}")
            continue

    driver.close()
    print("\n🎉 Fast-track CISA-NVD synchronization loop complete!")

if __name__ == "__main__":
    fast_track_cisa_nvd_sync()

In [45]:
def execute_database_text_mining_bridge(driver):
    """
    Scans description text inside your existing Software, Technique, and Procedure 
    nodes for 'CVE-YYYY-XXXX' values and builds clean direct graph linkages.
    """
    print("静态 Text-Mining Bridge Initiated...")
    
    with driver.session() as session:
        # 1. LINK SOFTWARE TO CVEs
        print("  → Analyzing Software descriptions for matching CVE patterns...")
        software_query = """
        MATCH (so:Software)
        WHERE so.description =~ '.*(?i)CVE-\\\\d{4}-\\\\d{4,7}.*'
        MATCH (c:CVE)
        WHERE so.description CONTAINS c.id
        MERGE (so)-[r:WEAPONIZES]->(c)
        RETURN count(r) AS sw_links
        """
        sw_count = session.run(software_query).single()["sw_links"]
        print(f"     ✅ Created {sw_count} direct (:WEAPONIZES) edges.")

        # 2. LINK INTERMEDIATE PROCEDURE NODES TO CVEs
        print("  → Analyzing intermediate Procedure nodes for execution footprints...")
        procedure_query = """
        MATCH (p:Procedure)
        WHERE p.description =~ '.*(?i)CVE-\\\\d{4}-\\\\d{4,7}.*'
        MATCH (c:CVE)
        WHERE p.description CONTAINS c.id
        MERGE (p)-[r:TARGETS_VULNERABILITY]->(c)
        RETURN count(r) AS proc_links
        """
        proc_count = session.run(procedure_query).single()["proc_links"]
        print(f"     ✅ Created {proc_count} direct (:TARGETS_VULNERABILITY) edges.")

        # 3. BUILD INFERRED TRANSITIVE TECHNIQUE MAPS (Separated to fix Cypher Aggregation Error)
        print("  → Computing transitive behavioral paths to Techniques...")
        
        # Path A: Via Software Weaponization (Software links to CVE and targets a Technique)
        path_a_query = """
        MATCH (t:Technique)<-[:TARGETS]-(p:Procedure)<-[:EXECUTED]-(so:Software)-[:WEAPONIZES]->(c:CVE)
        MERGE (c)-[r:INFERRED_MAPS_TO]->(t)
        SET r.mechanism = "via_software_weaponization", r.confidence = "high"
        RETURN count(r) AS count_a
        """
        tech_count_a = session.run(path_a_query).single()["count_a"]
        print(f"     ✅ Created {tech_count_a} edges via Software Weaponization paths.")
        
        # Path B: Via Direct Procedure Target Descriptions (Procedure notes target a Technique and describe a CVE)
        path_b_query = """
        MATCH (t:Technique)<-[:TARGETS]-(p:Procedure)-[:TARGETS_VULNERABILITY]->(c:CVE)
        MERGE (c)-[r:INFERRED_MAPS_TO]->(t)
        SET r.mechanism = "via_procedure_description", r.confidence = "high"
        RETURN count(r) AS count_b
        """
        tech_count_b = session.run(path_b_query).single()["count_b"]
        print(f"     ✅ Created {tech_count_b} edges via direct Procedure Context patterns.")
        
        total_inferred = tech_count_a + tech_count_b
        print(f"     🚀 Total active (:INFERRED_MAPS_TO) Technique connections: {total_inferred}")
        
        return sw_count + proc_count + total_inferred

if __name__ == "__main__":
    start_time = time.time()
    
    # Instantiate database connection using your exact properties
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    # Run the structural connector
    execute_database_text_mining_bridge(driver)
    
    driver.close()
    print(f"\n🎉 Graph fusion complete! Execution time: {round(time.time() - start_time, 2)} seconds.")

静态 Text-Mining Bridge Initiated...
  → Analyzing Software descriptions for matching CVE patterns...
     ✅ Created 4 direct (:WEAPONIZES) edges.
  → Analyzing intermediate Procedure nodes for execution footprints...
     ✅ Created 227 direct (:TARGETS_VULNERABILITY) edges.
  → Computing transitive behavioral paths to Techniques...
     ✅ Created 49 edges via Software Weaponization paths.
     ✅ Created 227 edges via direct Procedure Context patterns.
     🚀 Total active (:INFERRED_MAPS_TO) Technique connections: 276

🎉 Graph fusion complete! Execution time: 0.7 seconds.


---
## Cell 5 — Graph health check

In [5]:
CHECKS = [
    # Node counts
    ("Tactics",      "MATCH (n:Tactic)    RETURN count(n) AS cnt"),
    ("Techniques",   "MATCH (n:Technique) WHERE NOT n.is_subtechnique RETURN count(n) AS cnt"),
    ("Sub-techniques","MATCH (n:Technique) WHERE n.is_subtechnique RETURN count(n) AS cnt"),
    ("Groups",       "MATCH (n:Group)     RETURN count(n) AS cnt"),
    ("Software",     "MATCH (n:Software)  RETURN count(n) AS cnt"),
    ("CAPEC",        "MATCH (n:CAPEC)     RETURN count(n) AS cnt"),
    ("CWE",          "MATCH (n:CWE)       RETURN count(n) AS cnt"),
    ("CVE",          "MATCH (n:CVE)       RETURN count(n) AS cnt"),
    # Relationship counts
    ("BELONGS_TO (Technique→Tactic)",   "MATCH ()-[r:BELONGS_TO]->() RETURN count(r) AS cnt"),
    ("HAS_SUBTECHNIQUE",                 "MATCH ()-[r:HAS_SUBTECHNIQUE]->() RETURN count(r) AS cnt"),
    ("USES (Group/SW → Technique)",      "MATCH ()-[r:USES]->() RETURN count(r) AS cnt"),
    ("MAPS_TO_TECHNIQUE (CAPEC→ATT&CK)","MATCH ()-[r:MAPS_TO_TECHNIQUE]->() RETURN count(r) AS cnt"),
    ("TARGETS_WEAKNESS (CAPEC→CWE)",     "MATCH ()-[r:TARGETS_WEAKNESS]->() RETURN count(r) AS cnt"),
    ("CHILD_OF (CAPEC hierarchy)",       "MATCH ()-[r:CHILD_OF]->() RETURN count(r) AS cnt"),
    ("CAUSE_WEAKNESS (CVE→CWE)",         "MATCH ()-[r:CAUSE_WEAKNESS]->() RETURN count(r) AS cnt"),
    # Orphan checks
    ("Orphaned CVE  (no CWE)",  "MATCH (v:CVE) WHERE NOT (v)-[:CAUSE_WEAKNESS]->() RETURN count(v) AS cnt"),
    ("Orphaned CWE  (no edges)","MATCH (w:CWE) WHERE NOT ()-[]-(w) RETURN count(w) AS cnt"),
    ("Orphaned CAPEC (no ATT&CK link)","MATCH (ca:CAPEC) WHERE NOT (ca)-[:MAPS_TO_TECHNIQUE]->() RETURN count(ca) AS cnt"),
    ("Truly isolated CAPEC (no ATT&CK AND no CWE)", "MATCH (ca:CAPEC) WHERE NOT (ca)-[:MAPS_TO_TECHNIQUE]->() AND NOT (ca)-[:TARGETS_WEAKNESS]->() RETURN count(ca) AS cnt"),
]

print(f"{'Metric':<42} {'Count':>8}")
print("-" * 52)
with driver.session() as s:
    for label, query in CHECKS:
        result = s.run(query).single()
        cnt = result['cnt'] if result else 0
        flag = "  ⚠️" if 'Orphaned' in label and cnt > 0 else ""
        print(f"{label:<42} {cnt:>8}{flag}")

print("\n✅ Health check complete.")

Metric                                        Count
----------------------------------------------------
Tactics                                          28
Techniques                                      216
Sub-techniques                                  475
Groups                                          187
Software                                        784
CAPEC                                           615
CWE                                             338
CVE                                              53
BELONGS_TO (Technique→Tactic)                   887
HAS_SUBTECHNIQUE                                477
USES (Group/SW → Technique)                   16102
MAPS_TO_TECHNIQUE (CAPEC→ATT&CK)                399
TARGETS_WEAKNESS (CAPEC→CWE)                   1262
CHILD_OF (CAPEC hierarchy)                      533
CAUSE_WEAKNESS (CVE→CWE)                          7
Orphaned CVE  (no CWE)                           51  ⚠️
Orphaned CWE  (no edges)                          0
Orphane

In [47]:
import time
from neo4j import GraphDatabase

# --- YOUR EXISTING CONFIGURATION ---
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "kg_mitre_v1.1")

def execute_local_graph_cwe_technique_bridge(driver):
    print("🚀 Initiating Intra-Graph Structural Extraction Loop...")
    
    with driver.session() as session:
        # Step A: Scan Technique descriptions for any 'CWE-XXXX' mentions and link them directly
        print("  → Mining Technique nodes for raw CWE property references...")
        cwe_extraction_query = """
        MATCH (t:Technique)
        WHERE t.description =~ '.*(?i)CWE-\\\\d{1,5}.*'
        MATCH (w:CWE)
        WHERE t.description CONTAINS w.id
        MERGE (w)-[r:MAPPED_TO_TECHNIQUE]->(t)
        SET r.source = "IntraGraph_Description_Extraction",
            r.confidence = "high"
        RETURN count(r) AS mapped_links
        """
        mapped_links = session.run(cwe_extraction_query).single()["mapped_links"]
        print(f"     ✅ Formed {mapped_links} deterministic (:MAPPED_TO_TECHNIQUE) edges.")

        # Step B: Transitively carry those connections down to the individual CVE nodes
        print("  → Compounding structural pathways down to the CVE layer...")
        inference_query = """
        MATCH (c:CVE)-[:IDENTIFIES_WEAKNESS]->(w:CWE)-[:MAPPED_TO_TECHNIQUE]->(t:Technique)
        MERGE (c)-[r:INFERRED_MAPS_TO]->(t)
        SET r.mechanism = "via_local_cwe_extraction_path",
            r.confidence = "high"
        RETURN count(r) AS total_inferred
        """
        tech_count = session.run(inference_query).single()["total_inferred"]
        print(f"     ✅ Created {tech_count} high-fidelity transitive (:INFERRED_MAPS_TO) edges.")
        
        # Step C: Propagate vulnerability context back up to Procedure contexts to optimize logs
        print("  → Recovering coverage for intermediate operational Procedure nodes...")
        proc_query = """
        MATCH (t:Technique)<-[:TARGETS]-(p:Procedure)
        MATCH (c:CVE)-[:INFERRED_MAPS_TO]->(t)
        MERGE (p)-[r:TARGETS_VULNERABILITY]->(c)
        RETURN count(r) AS proc_linked
        """
        proc_count = session.run(proc_query).single()["proc_linked"]
        print(f"     ✅ Connected {proc_count} distinct logging Procedure contexts.")
        
        return mapped_links, tech_count, proc_count

if __name__ == "__main__":
    start_time = time.time()
    driver = GraphDatabase.driver(URI, auth=AUTH)
    
    execute_local_graph_cwe_technique_bridge(driver)
    
    driver.close()
    print(f"\n🎉 Graph fusion completed successfully in {round(time.time() - start_time, 2)} seconds!")

🚀 Initiating Intra-Graph Structural Extraction Loop...
  → Mining Technique nodes for raw CWE property references...
     ✅ Formed 0 deterministic (:MAPPED_TO_TECHNIQUE) edges.
  → Compounding structural pathways down to the CVE layer...
     ✅ Created 0 high-fidelity transitive (:INFERRED_MAPS_TO) edges.
  → Recovering coverage for intermediate operational Procedure nodes...
     ✅ Connected 12970 distinct logging Procedure contexts.

🎉 Graph fusion completed successfully in 0.42 seconds!


---
## Cell 6 — Example queries (read-only, run anytime)

These are starter Cypher queries to explore your graph.

In [7]:
SAMPLE_QUERIES = {

    "Full chain: CVE → CWE → CAPEC → Technique → Tactic": """
        MATCH (v:CVE)-[:CAUSE_WEAKNESS]->(w:CWE)
              <-[:TARGETS_WEAKNESS]-(ca:CAPEC)
              -[:MAPS_TO_TECHNIQUE]->(t:Technique)
              -[:BELONGS_TO]->(ta:Tactic)
        RETURN v.id, w.id, ca.id, ca.name, t.mitre_id, t.name, ta.name
        LIMIT 10
    """,

    "Groups using a specific technique (e.g. T1059 — Command Scripting)": """
        MATCH (g:Group)-[:USES]->(t:Technique {mitre_id: 'T1059'})
        RETURN g.name, g.mitre_id
        ORDER BY g.name
    """,

    "Top 10 most-used techniques across all groups": """
        MATCH (g:Group)-[:USES]->(t:Technique)
        RETURN t.mitre_id, t.name, count(g) AS group_count
        ORDER BY group_count DESC
        LIMIT 10
    """,

    "Software used by a group (e.g. APT29)": """
        MATCH (g:Group {name: 'APT29'})-[:USES]->(s:Software)
        RETURN s.name, s.type, s.mitre_id
    """,

    "Techniques under Credential Access tactic": """
        MATCH (t:Technique)-[:BELONGS_TO]->(ta:Tactic {short_name: 'credential-access'})
        WHERE NOT t.is_subtechnique
        RETURN t.mitre_id, t.name
        ORDER BY t.mitre_id
    """,

    "CAPEC→CWE→CVE for a given technique": """
        MATCH (ca:CAPEC)-[:MAPS_TO_TECHNIQUE]->(t:Technique {mitre_id: 'T1134'})
        OPTIONAL MATCH (ca)-[:TARGETS_WEAKNESS]->(w:CWE)
        OPTIONAL MATCH (v:CVE)-[:CAUSE_WEAKNESS]->(w)
        RETURN t.name, ca.id, ca.name, w.id, v.id
    """,
}

with driver.session() as s:
    for title, query in SAMPLE_QUERIES.items():
        print(f"\n{'='*60}")
        print(f"  {title}")
        print('='*60)
        rows = s.run(query).data()
        if rows:
            for row in rows:
                print(row)
        else:
            print("  (no results)")


  Full chain: CVE → CWE → CAPEC → Technique → Tactic
{'v.id': 'CVE-2023-23397', 'w.id': 'CWE-20', 'ca.id': 'CAPEC-13', 'ca.name': 'Subverting Environment Variable Values', 't.mitre_id': 'T1574.007', 't.name': 'Path Interception by PATH Environment Variable', 'ta.name': 'Persistence'}
{'v.id': 'CVE-2023-23397', 'w.id': 'CWE-20', 'ca.id': 'CAPEC-13', 'ca.name': 'Subverting Environment Variable Values', 't.mitre_id': 'T1574.007', 't.name': 'Path Interception by PATH Environment Variable', 'ta.name': 'Privilege Escalation'}
{'v.id': 'CVE-2023-23397', 'w.id': 'CWE-20', 'ca.id': 'CAPEC-13', 'ca.name': 'Subverting Environment Variable Values', 't.mitre_id': 'T1574.007', 't.name': 'Path Interception by PATH Environment Variable', 'ta.name': 'Defense Evasion'}
{'v.id': 'CVE-2023-23397', 'w.id': 'CWE-20', 'ca.id': 'CAPEC-13', 'ca.name': 'Subverting Environment Variable Values', 't.mitre_id': 'T1562', 't.name': 'Impair Defenses', 'ta.name': 'Defense Evasion'}
{'v.id': 'CVE-2023-23397', 'w.id': '